In [ ]:
!pip install -q transformers datasets
import tensorflow as tf
from transformers import ViTImageProcessor, TFViTModel
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
dataset = load_dataset("tonyassi/celebrity-1000", split="train")
selected_old_ids = range(10)

def filter_10(example):
    return example["label"] in selected_old_ids

dataset = dataset.filter(filter_10)
print("dataset size after filtering:", len(dataset))

In [ ]:
dataset[0]['image']

In [ ]:
print(len(selected_old_ids))

In [ ]:
dataset.shape

In [ ]:
def preprocess(batch):
    inputs = processor(images=batch["image"], return_tensors="np")
    return {"pixel_values": inputs["pixel_values"]}

In [ ]:
from transformers import ViTImageProcessor, TFViTForImageClassification

MODEL_NAME = "google/vit-base-patch16-224-in21k"

embedder = TFViTModel.from_pretrained(MODEL_NAME, from_pt=True)

processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

In [ ]:
dataset = dataset.map(preprocess, batched=True)

In [ ]:
ds = dataset.with_format("tensorflow")
tf_ds = dataset.to_tf_dataset(
    columns=["pixel_values"],
    label_cols=["label"],
    batch_size=64,
    shuffle=False
)

In [ ]:
all_embeddings = []

for batch, labels in tf_ds:
    outputs = embedder(batch)
    emb = outputs.pooler_output   # (batch, 768)
    all_embeddings.append(emb)

In [ ]:
all_embeddings = np.concatenate(all_embeddings, axis=0)  # (N, 768)
all_embeddings.shape

In [ ]:
dataset

In [ ]:
emb_list = [emb for emb in all_embeddings]
dataset = dataset.add_column("embedding", emb_list)

In [ ]:
dataset

In [ ]:
embeddings = np.array([e for e in dataset['embedding']])
labels = np.array([l for l in dataset['label']])

indices = np.arange(len(embeddings))
np.random.shuffle(indices)

embeddings_shuffled = embeddings[indices]
labels_shuffled = labels[indices]

pd.DataFrame(embeddings_shuffled).to_csv('embeddings.csv', index=False, header=False)
pd.DataFrame(labels_shuffled).to_csv('labels.csv', index=False, header=False)